# Amazon ML Challenge — Memory-Safe Exhaustive EDA

Designed for very large TSV files (millions of rows) on Kaggle.

The notebook deliberately avoids:
- loading all six source tables into memory
- creating full normalized/transliterated copies
- full Cartesian joins
- full-pair string comparisons

It instead uses:
- streaming/chunked TSV reads
- bounded deterministic samples
- full-data cheap statistics
- full-data country / placeholder / missingness counts
- bounded top-frequency collision analysis
- complete ground-truth analysis
- sampled positive-pair resolution
- sampled hard-negative analysis
- multilingual/script and Unicode diagnostics
- name/address similarity comparisons
- candidate/blocking-style diagnostics on sampled data
- train/test distribution drift
- a final focus report containing observed EDA issues

This notebook is for EDA only. It does not choose a matching model.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import os
import sys
import time
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]
BASENAMES = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv"
]

CHUNK_SIZE = 100_000
SAMPLE_PER_SOURCE = 20_000
POSITIVE_PAIR_SAMPLE = 20_000
HARD_NEGATIVE_SAMPLE = 20_000
TOP_K = 50
N_THREADS = min(8, max(2, os.cpu_count() or 2))

try:
    from rapidfuzz.fuzz import ratio, WRatio, token_sort_ratio, token_set_ratio
    HAS_RAPIDFUZZ = True
except Exception:
    HAS_RAPIDFUZZ = False
    from difflib import SequenceMatcher

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

print("Python:", sys.version.split()[0])
print("CPU cores:", os.cpu_count())
print("Threads:", N_THREADS)
print("RapidFuzz:", HAS_RAPIDFUZZ)

Python: 3.12.13
CPU cores: 4
Threads: 4
RapidFuzz: False


In [3]:
def find_file(name):
    found = []
    for root in SEARCH_ROOTS:
        if root.exists():
            try:
                found.extend(root.rglob(name))
            except Exception:
                pass
    found = sorted(set(x.resolve() for x in found))
    return found[0] if found else None

paths = {name: find_file(name) for name in BASENAMES}
display(pd.DataFrame({"file": list(paths), "path": [str(paths[x]) if paths[x] else None for x in paths]}))

missing = [x for x, p in paths.items() if p is None]
if missing:
    raise FileNotFoundError("Missing files: " + ", ".join(missing))

,file,path
0,train_source1.tsv,/kaggle/input/datasets/keyaanesss/datasettrain/train_source1.tsv
1,train_source2.tsv,/kaggle/input/datasets/keyaanesss/datasettrain/train_source2.tsv
2,train_source3.tsv,/kaggle/input/datasets/keyaanesss/datasettrain/train_source3.tsv
3,train_ground_truth.tsv,/kaggle/input/datasets/keyaanesss/datasettrain/train_ground_truth.tsv
4,test_source1.tsv,/kaggle/input/datasets/keyaanesss/datasettest/test_source1.tsv
5,test_source2.tsv,/kaggle/input/datasets/keyaanesss/datasettest/test_source2.tsv
6,test_source3.tsv,/kaggle/input/datasets/keyaanesss/datasettest/test_source3.tsv


## 1. Streaming full-data profile

In [4]:
SOURCE_COLS = ["entity_id", "business_name", "business_address", "country"]

def empty_profile():
    return {
        "rows": 0,
        "duplicate_rows_within_chunks": 0,
        "blank": Counter(),
        "placeholder": Counter(),
        "wrong_prefix": 0,
        "blank_id": 0,
        "id_duplicate_within_chunks": 0,
        "country_counts": Counter(),
        "name_len_sum": 0.0,
        "name_len_sq_sum": 0.0,
        "name_len_min": np.inf,
        "name_len_max": 0,
        "address_len_sum": 0.0,
        "address_len_sq_sum": 0.0,
        "address_len_min": np.inf,
        "address_len_max": 0,
        "sample_parts": []
    }

PLACEHOLDERS = {"na", "n/a", "null", "none", "unknown", "unk", "-", "--", "missing", "?"}

def update_profile(profile, chunk, expected_prefix, sample_rate, rng):
    profile["rows"] += len(chunk)
    profile["duplicate_rows_within_chunks"] += int(chunk.duplicated().sum())
    for col in SOURCE_COLS:
        s = chunk[col].astype(str)
        stripped = s.str.strip()
        profile["blank"][col] += int(stripped.eq("").sum())
        vals = stripped.str.casefold()
        profile["placeholder"].update(vals[vals.isin(PLACEHOLDERS)].value_counts().to_dict())
    ids = chunk["entity_id"].astype(str)
    profile["blank_id"] += int(ids.str.strip().eq("").sum())
    profile["wrong_prefix"] += int((~ids.str.startswith(expected_prefix)).sum())
    profile["id_duplicate_within_chunks"] += int(ids.duplicated().sum())
    profile["country_counts"].update(chunk["country"].astype(str).str.strip().str.casefold().value_counts().to_dict())
    nl = chunk["business_name"].astype(str).str.len().to_numpy(dtype=np.float64)
    al = chunk["business_address"].astype(str).str.len().to_numpy(dtype=np.float64)
    profile["name_len_sum"] += nl.sum()
    profile["name_len_sq_sum"] += np.square(nl).sum()
    profile["name_len_min"] = min(profile["name_len_min"], nl.min() if len(nl) else np.inf)
    profile["name_len_max"] = max(profile["name_len_max"], int(nl.max()) if len(nl) else 0)
    profile["address_len_sum"] += al.sum()
    profile["address_len_sq_sum"] += np.square(al).sum()
    profile["address_len_min"] = min(profile["address_len_min"], al.min() if len(al) else np.inf)
    profile["address_len_max"] = max(profile["address_len_max"], int(al.max()) if len(al) else 0)

    if len(chunk) > 0 and sample_rate > 0:
        take_n = min(len(chunk), max(1, int(np.ceil(len(chunk) * sample_rate))))
        sample_idx = rng.choice(len(chunk), size=take_n, replace=False)
        profile["sample_parts"].append(chunk.iloc[sample_idx][SOURCE_COLS].copy())

def stream_profile(path, expected_prefix, target_sample):
    first = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False, nrows=5)
    if list(first.columns) != SOURCE_COLS:
        raise ValueError(f"{path.name} columns are {list(first.columns)}, expected {SOURCE_COLS}")
    total_rows_est = 0
    profile = empty_profile()
    local_rng = np.random.default_rng(SEED)
    for chunk in pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False, chunksize=CHUNK_SIZE):
        total_rows_est += len(chunk)
        current_rate = min(0.01, max(0.001, target_sample / max(1, 20 * target_sample)))
        update_profile(profile, chunk, expected_prefix, current_rate, local_rng)
    if profile["sample_parts"]:
        sample = pd.concat(profile["sample_parts"], ignore_index=True)
        if len(sample) > target_sample:
            sample = sample.sample(target_sample, random_state=SEED).reset_index(drop=True)
    else:
        sample = pd.DataFrame(columns=SOURCE_COLS)
    profile["sample"] = sample
    del profile["sample_parts"]
    return profile

t0 = time.perf_counter()
profiles = {}
prefixes = {
    "train_source1": "S1-", "train_source2": "S2-", "train_source3": "S3-",
    "test_source1": "S1-", "test_source2": "S2-", "test_source3": "S3-"
}

for key in prefixes:
    profiles[key] = stream_profile(paths[key + ".tsv"], prefixes[key], SAMPLE_PER_SOURCE)
    print(key, profiles[key]["rows"], "rows")

print("Profile minutes:", round((time.perf_counter() - t0) / 60, 2))

train_source1 2206821 rows
train_source2 5034616 rows
train_source3 5285603 rows
test_source1 1732544 rows
test_source2 4887273 rows
test_source3 5082316 rows
Profile minutes: 4.06


In [5]:
profile_rows = []
for name, p in profiles.items():
    n = p["rows"]
    profile_rows.append({
        "dataset": name,
        "rows": n,
        "blank_name_pct": 100 * p["blank"]["business_name"] / max(1, n),
        "blank_address_pct": 100 * p["blank"]["business_address"] / max(1, n),
        "blank_country_pct": 100 * p["blank"]["country"] / max(1, n),
        "blank_id_n": p["blank_id"],
        "wrong_prefix_n": p["wrong_prefix"],
        "duplicate_rows_within_chunks_n": p["duplicate_rows_within_chunks"],
        "duplicate_ids_within_chunks_n": p["id_duplicate_within_chunks"],
        "name_mean_len": p["name_len_sum"] / max(1, n),
        "name_min_len": p["name_len_min"] if np.isfinite(p["name_len_min"]) else 0,
        "name_max_len": p["name_len_max"],
        "address_mean_len": p["address_len_sum"] / max(1, n),
        "address_min_len": p["address_len_min"] if np.isfinite(p["address_len_min"]) else 0,
        "address_max_len": p["address_len_max"],
        "sample_rows": len(p["sample"])
    })

full_profile = pd.DataFrame(profile_rows)
display(full_profile)

,dataset,rows,blank_name_pct,blank_address_pct,blank_country_pct,blank_id_n,wrong_prefix_n,duplicate_rows_within_chunks_n,duplicate_ids_within_chunks_n,name_mean_len,name_min_len,name_max_len,address_mean_len,address_min_len,address_max_len,sample_rows
0,train_source1,2206821,0.0,0.000000,0.0,0,0,0,0,24.034403,3.0,105,52.066213,11.0,256,20000
1,train_source2,5034616,0.0,3.356105,0.0,0,0,0,0,25.103575,2.0,104,46.225909,0.0,249,20000
2,train_source3,5285603,0.0,3.328211,0.0,0,0,0,0,25.202144,2.0,123,46.714437,0.0,240,20000
3,test_source1,1732544,0.0,0.000000,0.0,0,0,0,0,23.836426,3.0,92,57.213180,11.0,268,17326
4,test_source2,4887273,0.0,2.647857,0.0,0,0,0,0,25.703646,2.0,102,50.413406,0.0,269,20000
5,test_source3,5082316,0.0,2.677874,0.0,0,0,0,0,25.655194,2.0,103,48.740471,0.0,267,20000


## 2. Country and placeholder diagnostics on the full data

In [6]:
country_rows = []
for name, p in profiles.items():
    for country, count in p["country_counts"].most_common(TOP_K):
        country_rows.append({"dataset": name, "country": country, "count": count, "pct": 100 * count / max(1, p["rows"])})
display(pd.DataFrame(country_rows))

placeholder_rows = []
for name, p in profiles.items():
    for value, count in p["placeholder"].most_common():
        placeholder_rows.append({"dataset": name, "placeholder": value, "count": count})
display(pd.DataFrame(placeholder_rows) if placeholder_rows else pd.DataFrame(columns=["dataset","placeholder","count"]))

,dataset,country,count,pct
0,train_source1,us,1323633,59.979174
1,train_source1,india,883188,40.020826
2,train_source2,us,3016817,59.921492
3,train_source2,india,2017799,40.078508
4,train_source3,us,3170056,59.975295
5,train_source3,india,2115547,40.024705
6,test_source1,india,809986,46.751251
7,test_source1,us,663106,38.273545
8,test_source1,france,259452,14.975204
9,test_source2,india,2312565,47.318106


,dataset,placeholder,count
0,train_source2,na,5
1,train_source3,na,18
2,test_source2,na,49
3,test_source3,na,60
4,test_source3,null,1


## 3. Sample normalization and multilingual diagnostics

In [7]:
def basic_norm(x):
    x = unicodedata.normalize("NFKC", str(x)).casefold()
    x = "".join(ch for ch in unicodedata.normalize("NFKD", x) if not unicodedata.combining(ch))
    x = re.sub(r"\s+", " ", x).strip()
    return x

def alnum_norm(x):
    x = basic_norm(x)
    x = re.sub(r"[^\w\s]", " ", x, flags=re.UNICODE)
    return re.sub(r"\s+", " ", x).strip()

def translit_norm(x):
    x = unicodedata.normalize("NFKC", str(x))
    try:
        from unidecode import unidecode
        x = unidecode(x)
    except Exception:
        pass
    x = x.casefold()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

def script_of_char(ch):
    if ch.isdigit():
        return "DIGIT"
    try:
        name = unicodedata.name(ch)
    except Exception:
        return "UNKNOWN"
    markers = [
        ("LATIN","LATIN"),("GREEK","GREEK"),("CYRILLIC","CYRILLIC"),("ARABIC","ARABIC"),
        ("HEBREW","HEBREW"),("DEVANAGARI","DEVANAGARI"),("BENGALI","BENGALI"),
        ("GURMUKHI","GURMUKHI"),("GUJARATI","GUJARATI"),("ORIYA","ORIYA"),
        ("TAMIL","TAMIL"),("TELUGU","TELUGU"),("KANNADA","KANNADA"),("MALAYALAM","MALAYALAM"),
        ("SINHALA","SINHALA"),("THAI","THAI"),("KHMER","KHMER"),("MYANMAR","MYANMAR"),
        ("ARMENIAN","ARMENIAN"),("GEORGIAN","GEORGIAN"),("HIRAGANA","HIRAGANA"),
        ("KATAKANA","KATAKANA"),("HANGUL","HANGUL"),("CJK","CJK")
    ]
    for marker, label in markers:
        if marker in name:
            return label
    if ch.isalpha():
        return "OTHER_LETTER"
    return "NON_LETTER"

def text_metrics(x):
    x = str(x)
    scripts = Counter(script_of_char(ch) for ch in x if not ch.isspace())
    letter_scripts = [k for k,v in scripts.items() if v and k not in {"DIGIT","NON_LETTER"}]
    dominant = max(letter_scripts, key=lambda k: scripts[k]) if letter_scripts else "NONE"
    non_ascii = sum(ord(ch) > 127 for ch in x)
    controls = sum(unicodedata.category(ch) in {"Cc","Cf"} for ch in x)
    combining = sum(unicodedata.combining(ch) > 0 for ch in x)
    weird_ws = sum(ch in "\t\n\r\u00a0\u200b\u200c\u200d" for ch in x)
    digits = sum(ch.isdigit() for ch in x)
    letters = sum(ch.isalpha() for ch in x)
    punctuation = sum(unicodedata.category(ch).startswith("P") for ch in x)
    return {
        "length": len(x),
        "tokens": len(re.findall(r"\S+", x)),
        "digits": digits,
        "letters": letters,
        "punctuation": punctuation,
        "non_ascii": non_ascii,
        "control": controls,
        "combining": combining,
        "weird_ws": weird_ws,
        "dominant_script": dominant,
        "script_count": len(letter_scripts),
        "mixed_script": len(letter_scripts) > 1
    }

def threaded_metrics(values):
    values = [str(x) for x in values]
    chunks = np.array_split(np.arange(len(values)), max(1, min(N_THREADS * 2, len(values))))
    def worker(idx):
        return [text_metrics(values[i]) for i in idx.tolist()]
    with ThreadPoolExecutor(max_workers=N_THREADS) as ex:
        parts = list(ex.map(worker, chunks))
    return pd.DataFrame([item for part in parts for item in part])

sample_dfs = {name: p["sample"].copy() for name,p in profiles.items()}

sample_script_rows = []
script_counts = []
for name, df in sample_dfs.items():
    if len(df) == 0:
        continue
    for field in ["business_name","business_address"]:
        metrics = threaded_metrics(df[field].tolist())
        sample_script_rows.append({
            "dataset":name,
            "field":field,
            "sample_n":len(df),
            "mixed_script_pct":100*metrics["mixed_script"].mean(),
            "non_ascii_pct":100*metrics["non_ascii"].gt(0).mean(),
            "control_pct":100*metrics["control"].gt(0).mean(),
            "combining_pct":100*metrics["combining"].gt(0).mean(),
            "weird_whitespace_pct":100*metrics["weird_ws"].gt(0).mean()
        })
        q = metrics["dominant_script"].value_counts().rename_axis("script").reset_index(name="count")
        q.insert(0,"dataset",name)
        q.insert(1,"field",field)
        script_counts.append(q)

display(pd.DataFrame(sample_script_rows))
display(pd.concat(script_counts, ignore_index=True).sort_values(["dataset","field","count"], ascending=[True,True,False]).head(200) if script_counts else pd.DataFrame(columns=["dataset","field","script","count"]))

,dataset,field,sample_n,mixed_script_pct,non_ascii_pct,control_pct,combining_pct,weird_whitespace_pct
0,train_source1,business_name,20000,0.000000,0.000000,0.000000,0.000,0.000
1,train_source1,business_address,20000,0.015000,0.010000,0.015000,0.000,0.000
2,train_source2,business_name,20000,0.410000,15.690000,0.510000,9.495,0.510
3,train_source2,business_address,20000,9.415000,9.415000,0.030000,7.165,0.000
4,train_source3,business_name,20000,0.695000,11.370000,0.340000,5.215,0.340
5,train_source3,business_address,20000,8.865000,8.865000,0.015000,6.640,0.000
6,test_source1,business_name,17326,0.000000,2.504906,0.000000,0.000,0.000
7,test_source1,business_address,17326,0.028858,4.080573,0.028858,0.000,0.000
8,test_source2,business_name,20000,0.350000,18.745000,0.645000,10.935,0.645
9,test_source2,business_address,20000,11.550000,14.600000,0.030000,8.580,0.000


,dataset,field,script,count
36,test_source1,business_address,LATIN,17326
35,test_source1,business_name,LATIN,17326
47,test_source2,business_address,LATIN,19409
48,test_source2,business_address,NONE,574
49,test_source2,business_address,DEVANAGARI,14
...,...,...,...,...
25,train_source3,business_name,BENGALI,54
26,train_source3,business_name,MALAYALAM,36
27,train_source3,business_name,GURMUKHI,25
28,train_source3,business_name,ORIYA,18


## 4. Sample collision and normalization analysis

In [8]:
norm_samples = {}
collision_rows = []

for name, df in sample_dfs.items():
    x = df.copy()
    for field in ["business_name","business_address","country"]:
        x[field+"_basic"] = x[field].map(basic_norm)
        x[field+"_alnum"] = x[field].map(alnum_norm)
        if field != "country":
            x[field+"_translit"] = x[field].map(translit_norm)
    norm_samples[name] = x

    for rep in ["business_name_basic","business_name_alnum","business_address_basic","business_address_alnum"]:
        counts = x[rep].value_counts()
        nonblank = counts[counts.index.astype(str).str.strip() != ""]
        collision_rows.append({
            "dataset":name,
            "representation":rep,
            "sample_unique":len(nonblank),
            "groups_2plus":int((nonblank>=2).sum()),
            "groups_10plus":int((nonblank>=10).sum()),
            "rows_in_2plus_groups":int(nonblank[nonblank>=2].sum()),
            "max_collision":int(nonblank.max()) if len(nonblank) else 0
        })

    for rep in ["business_name_translit","business_address_translit"]:
        counts = x[rep].value_counts()
        nonblank = counts[counts.index.astype(str).str.strip() != ""]
        collision_rows.append({
            "dataset":name,
            "representation":rep,
            "sample_unique":len(nonblank),
            "groups_2plus":int((nonblank>=2).sum()),
            "groups_10plus":int((nonblank>=10).sum()),
            "rows_in_2plus_groups":int(nonblank[nonblank>=2].sum()),
            "max_collision":int(nonblank.max()) if len(nonblank) else 0
        })

display(pd.DataFrame(collision_rows))

,dataset,representation,sample_unique,groups_2plus,groups_10plus,rows_in_2plus_groups,max_collision
0,train_source1,business_name_basic,19456,477,0,1021,5
1,train_source1,business_name_alnum,19425,502,0,1077,5
2,train_source1,business_address_basic,19990,10,0,20,2
3,train_source1,business_address_alnum,19990,10,0,20,2
4,train_source1,business_name_translit,19425,502,0,1077,5
5,train_source1,business_address_translit,19990,10,0,20,2
6,train_source2,business_name_basic,19909,85,0,176,3
7,train_source2,business_name_alnum,19881,109,0,228,4
8,train_source2,business_address_basic,19317,4,0,8,2
9,train_source2,business_address_alnum,19316,5,0,10,2


In [9]:
def show_collision_examples(name, rep, n=15):
    df = norm_samples[name]
    x = df[df[rep].astype(str).str.strip() != ""].copy()
    counts = x[rep].value_counts()
    keys = counts[counts>=2].head(n).index
    cols = ["entity_id","business_name","business_address","country",rep]
    return x[x[rep].isin(keys)][cols].sort_values(rep)

for name in ["train_source1","train_source2","train_source3"]:
    print(name, "name transliteration collisions")
    display(show_collision_examples(name,"business_name_translit",15))
    print(name, "address transliteration collisions")
    display(show_collision_examples(name,"business_address_translit",15))

train_source1 name transliteration collisions


,entity_id,business_name,business_address,country,business_name_translit
9976,S1-76560335,Aditya It Private Limited,"Door No. 22/799, Sankar Nivas Beach North, Kollam, Kerala",India,aditya it private limited
185,S1-962353775,Aditya It Private Limited,"215 Veena Beena Shopping Centre, 2Nd Floor Opp Railway Station, Bandra(West), Mumbai, Mumbai City, Maharashtra",India,aditya it private limited
2374,S1-864418112,Aditya It Private Limited,"Kb-69043, C-|-Pm-11312, Hinganghat, Wardha, Maharashtra",India,aditya it private limited
12074,S1-294516121,Aditya It Private Limited,"Hyderabad, H. No-2-105/1/A Gopanpally Gachibowli, Telangana, Hyderabad",India,aditya it private limited
18399,S1-181298785,Cobalt LLC,"62060 Valley View Circle, Joshua Tree, CA",US,cobalt llc
14429,S1-686524100,Cobalt LLC,"City Of Cedarburg, WI, N48W6340 Elkhart Drive",US,cobalt llc
12991,S1-881196349,Cobalt LLC,"110 Meadowbrook Acres, Candler, NC",US,cobalt llc
9929,S1-655436871,Cobalt LLC,"262 Pilot Drive, Somerset, MA",US,cobalt llc
16180,S1-659720907,Dental Group,"512 Mable Mason Cove, Lavergne, TN",US,dental group
12330,S1-505079881,Dental Group,"123 Pk Drive, Brownsville, TX",US,dental group


train_source1 address transliteration collisions


,entity_id,business_name,business_address,country,business_address_translit
4236,S1-215213133,Great Network P.C.,"1001 Griswold Road, Unit 30, Phoenix, AZ",US,1001 griswold road unit 30 phoenix az
3659,S1-361459032,Kinard Pacific Molecular P.C.,"1001 Griswold Road, Unit 30, Phoenix, AZ",US,1001 griswold road unit 30 phoenix az
13243,S1-513060102,Osprey Jewelers Center,"1117 Sparrow Drive, Port Hueneme, CA",US,1117 sparrow drive port hueneme ca
933,S1-790072780,Vertex Origin Inc,"1117 Sparrow Drive, Port Hueneme, CA",US,1117 sparrow drive port hueneme ca
12045,S1-199018026,Willmon and Tate LLC,"1903 Rock Bluff Road, Plattsmouth, NE",US,1903 rock bluff road plattsmouth ne
164,S1-840168502,Modern Coastal Fargo Inc.,"1903 Rock Bluff Road, Plattsmouth, NE",US,1903 rock bluff road plattsmouth ne
337,S1-634010650,Zeix LLC,"316 Sugar Loaf Mountain Road, Chester, NY",US,316 sugar loaf mountain road chester ny
14026,S1-257430011,"Darcie D. Cohen, O.D.","316 Sugar Loaf Mountain Road, Chester, NY",US,316 sugar loaf mountain road chester ny
9377,S1-18320937,Kevoum Partners LLC,"4 Micajah Avenue, Plymouth, MA",US,4 micajah avenue plymouth ma
1368,S1-217296719,Metro Interiors Inc.,"4 Micajah Avenue, Plymouth, MA",US,4 micajah avenue plymouth ma


train_source2 name transliteration collisions


...

,entity_id,business_name,business_address,country,business_address_translit
416,S2-582295053,TOREY FEHR SUMMIT CONSTRUCTION,"101. SPRINGLINE DRIVE, CHESTERFIELD COUNTY, VA",US,101 springline drive chesterfield county va
10723,S2-768610826,Rocky Lion,"101 SPRINGLINE DRIVE, CHESTERFIELD COUNTY, VA",US,101 springline drive chesterfield county va
1422,S2-813108789,BUREAU OF AGING,"1106 Tamarack Way, CITY OF VERONA, WI",US,1106 tamarack way city of verona wi
10676,S2-459773140,Bureau of Aging Corp,"1106 TAMARACK WAY, CITY OF VERONA, WI",US,1106 tamarack way city of verona wi
426,S2-246771463,PACIFICPROGRAM.COM,"2906 65TH AVENUE COURT, GIG HAARBOR, WA",US,2906 65th avenue court gig haarbor wa
3727,S2-763422989,PPACIFIC.COM,"2906 65TH AVENUE COURT, GIG HAARBOR, WA",US,2906 65th avenue court gig haarbor wa
17900,S2-914035589,ರಾಮ್ ಪ್ರೊಡ್ಯೂಸರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"NO. B-3, PID NO. 64-146-03, K.E.B LAYOUT, BTM LAYOUT, 1ST STAGE, MAIN ROAD, BANNERGHATTA, BANGALORE, Karnataka",India,no b 3 pid no 64 146 03 k e b layout btm layout 1st stage main road bannerghatta bangalore karnataka
18240,S2-691990937,ರಾಮ್ ಪ್ರೊಡ್ಯೂಸರ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"NO. B-3, PID NO. 64-146-03, K.E.B LAYOUT, BTM LAYOUT, 1ST STAGE, MAIN ROAD, BANNERGHATTA, BANGALORE, Karnataka",India,no b 3 pid no 64 146 03 k e b layout btm layout 1st stage main road bannerghatta bangalore karnataka
4646,S2-127425046,Navdurga Solutions Partners Company,"OFFICE NO 169 DISMA TOWER, LOT NO 246, PANVEL, Maharashtra",India,office no 169 disma tower lot no 246 panvel maharashtra
11512,S2-390238384,Navdurga Solutions Partners Corporation,"OFFICE NO 169 DISMA TOWER, LOT NO 246, PANVEL, Maharashtra",India,office no 169 disma tower lot no 246 panvel maharashtra


train_source3 name transliteration collisions


,entity_id,business_name,business_address,country,business_name_translit
7283,S3-597200250,ACE HOSPITALITY,,India,ace hospitality
7212,S3-170660445,Ace Hospitality,"#4513 9nd Street, Lubbock, Texas",US,ace hospitality
16553,S3-19013219,Ace-Hospitality,"Door No 104C, Gera Imperium Rise, Mulshi, Pune, Haveli, MH",India,ace hospitality
1191,S3-108895711,Apex,"F-1902, Sunshine Helios, Sector-78, Noida, Gautam Buddha Nagar, Uttar Pradesh",India,apex
16659,S3-83243184,Apex,"1045 1440, South Jordan, Utah",US,apex
6973,S3-662795585,Apex,"16359 34th Avenue, Lake Forest Park, WA",US,apex
5236,S3-932293205,Dynamic,"H/O Shahada Khanam Ashiana Road Samanpura, Near B.V. Colleg, E, Patna, Bihar",India,dynamic
17221,S3-895775427,Dynamic,"No 65 First Floor Punitha, Anthoniyar 3Rd Street, Ponneri, Tiruvallur, Tamil Nadu",India,dynamic
16519,S3-440884739,Dynamic,"116 Mallory Tai Drive, Lincolnton, NC",US,dynamic
17141,S3-297145261,Eye Group,"##952 Stafford Hill Rd, Cheshire, Massachusetts",US,eye group


train_source3 address transliteration collisions


,entity_id,business_name,business_address,country,business_address_translit
271,S3-855956509,"Asher's First Jewelers,","006801 Limerick Ln, Edina, Minnesota",US,006801 limerick ln edina minnesota
13231,S3-858942329,"Asher's 1st Jewelers, Inc","006801 Limerick Ln, Edina, Minnesota",US,006801 limerick ln edina minnesota
15212,S3-812671529,coastalpegasussuperior.com,"11 Elm Park, Groveland CITY, Massachusetts",US,11 elm park groveland city massachusetts
7169,S3-176576071,Coasta1 Superior Corp Service,"11 Elm Park, Groveland CITY, Massachusetts",US,11 elm park groveland city massachusetts
1137,S3-759270786,Crystal Safe Ellington,"11411-D Farm Road 156, Repulic, Missouri",US,11411 d farm road 156 repulic missouri
2864,S3-261502067,Great Coalition,"11411-D Farm Road 156, Repulic, Missouri",US,11411 d farm road 156 repulic missouri
4671,S3-142565623,Dynamic Ltd Annuities,"##1571 Benson Airport Road, Benson, Arizona",US,1571 benson airport road benson arizona
11261,S3-930384332,Corp. Dynamic Annuities,"##1571 Benson Airport Road, Benson, Arizona",US,1571 benson airport road benson arizona
2968,S3-346536983,reed & sweet fact llc,"20900 County Rd, Wilmington, Illinois",US,20900 county rd wilmington illinois
13432,S3-398737417,Reed and Sweet,"20900 County Rd, Wilmington, Illinois",US,20900 county rd wilmington illinois


## 5. Edge-case flags on a stratified sample

In [10]:
def edge_flags(df):
    out=[]
    for r in df.itertuples(index=False):
        flags=[]
        name=str(r.business_name)
        addr=str(r.business_address)
        if not name.strip(): flags.append("name_missing")
        if not addr.strip(): flags.append("address_missing")
        if len(name.strip()) <= 2 and name.strip(): flags.append("name_very_short")
        if len(addr.strip()) <= 5 and addr.strip(): flags.append("address_very_short")
        if re.fullmatch(r"\d+", name.strip() or "X"): flags.append("name_numeric")
        if re.fullmatch(r"\d+", addr.strip() or "X"): flags.append("address_numeric")
        if re.fullmatch(r"[^\w\s]+", name.strip() or "X"): flags.append("name_punctuation_only")
        if re.fullmatch(r"[^\w\s]+", addr.strip() or "X"): flags.append("address_punctuation_only")
        nm=text_metrics(name)
        am=text_metrics(addr)
        if nm["mixed_script"]: flags.append("name_mixed_script")
        if am["mixed_script"]: flags.append("address_mixed_script")
        if nm["control"] or am["control"]: flags.append("control_characters")
        if nm["weird_ws"] or am["weird_ws"]: flags.append("weird_whitespace")
        if len(name) >= 150: flags.append("name_very_long")
        if len(addr) >= 400: flags.append("address_very_long")
        if name.strip() and translit_norm(name)=="" : flags.append("name_translit_empty")
        if addr.strip() and translit_norm(addr)=="" : flags.append("address_translit_empty")
        out.append("|".join(flags))
    return pd.Series(out)

edge_summary=[]
edge_examples=[]

for name,df in sample_dfs.items():
    flags = edge_flags(df)
    counts = Counter()
    for f in flags:
        if f:
            counts.update(f.split("|"))
    for issue,count in counts.most_common():
        edge_summary.append({"dataset":name,"issue":issue,"sample_count":count,"sample_pct":100*count/max(1,len(df))})
    mask=flags.ne("")
    if mask.any():
        ex=df.loc[mask,["entity_id","business_name","business_address","country"]].copy()
        ex.insert(0,"dataset",name)
        ex.insert(1,"issues",flags.loc[mask].values)
        edge_examples.append(ex.head(100))

display(pd.DataFrame(edge_summary).sort_values(["dataset","sample_count"],ascending=[True,False]) if edge_summary else pd.DataFrame(columns=["dataset","issue","sample_count","sample_pct"]))
display(pd.concat(edge_examples,ignore_index=True).head(500) if edge_examples else pd.DataFrame(columns=["dataset","issues","entity_id","business_name","business_address","country"]))

,dataset,issue,sample_count,sample_pct
16,test_source1,address_mixed_script,5,0.028858
17,test_source1,control_characters,5,0.028858
18,test_source2,address_mixed_script,2310,11.550000
19,test_source2,name_translit_empty,2182,10.910000
20,test_source2,address_missing,574,2.870000
21,test_source2,control_characters,135,0.675000
22,test_source2,weird_whitespace,129,0.645000
23,test_source2,name_mixed_script,70,0.350000
24,test_source2,name_very_short,39,0.195000
25,test_source3,address_mixed_script,2242,11.210000


,dataset,issues,entity_id,business_name,business_address,country
0,train_source1,address_mixed_script|control_characters,S1-705969341,Urban Gold Global Private Limited,"Hyderabad, M. No. 8-2-682/2/E, Road No. 12, Banjara Hills, Hyderabad, ZonahâS Crescent, Telangana",India
1,train_source1,address_mixed_script|control_characters,S1-924839198,Gar & Brothers Private Limited,"112 Â 2, Gaonthan, Padali, Ranjangaon, Parner, Parner, Ahmednagar, Maharashtra",India
2,train_source1,address_mixed_script|control_characters,S1-89156663,All Impex Private Limited,"405, 'CSwastik Disha Corporate Park, Kohinoor Compound L.B.S Marg, Opp.Shreyas Cinema, Ghatkopar, West, Mumbai, Mum...",India
3,train_source2,address_missing,S2-875545903,PO LLC-Fitness Regional,,US
4,train_source2,address_missing,S2-724068773,Waterford Vision,,US
...,...,...,...,...,...,...
403,test_source3,address_mixed_script,S3-100001745,Pvt Fortune Products (Ltd.),"116/218 Rawatpur Kanpur, उत्तर प्रदेश, Kanpur, Kanpur Nagar",India
404,test_source3,address_mixed_script,S3-620259306,Shakti Projects Provision Provision,"H.no ##200 172/5/5, Bangalore, ಕರ್ನಾಟಕ",India
405,test_source3,name_translit_empty,S3-287317340,માય ગ્લોબલ એન્જિનિયરિંગ પ્રાઇવેટ લિમિટેડ,"H.no 216 Ashwamegh Avenuenr Under Bridge Mithakhali, Ahmedabad, GJ",India
406,test_source3,address_mixed_script,S3-201938990,Mr Jaya & Sons Group Private Limited,"57/3002, Cochin, Ernakulam, കേരളം",India


## 6. Ground-truth structure

In [11]:
def split_matches(x):
    x=str(x).strip()
    return [] if not x else [z.strip() for z in x.split(",") if z.strip()]

gt=pd.read_csv(paths["train_ground_truth.tsv"],sep="\t",dtype=str,keep_default_na=False,na_filter=False)
expected_gt=["source1_entity_id","matched_entity_ids"]
if list(gt.columns)!=expected_gt:
    raise ValueError(f"Ground truth columns are {list(gt.columns)}, expected {expected_gt}")

gt["match_list"]=gt["matched_entity_ids"].map(split_matches)
gt["match_count"]=gt["match_list"].map(len)
gt["s2_count"]=gt["match_list"].map(lambda z:sum(x.startswith("S2-") for x in z))
gt["s3_count"]=gt["match_list"].map(lambda z:sum(x.startswith("S3-") for x in z))

print("Ground-truth rows:",len(gt))
print("Ground-truth duplicate S1 IDs:",int(gt["source1_entity_id"].duplicated().sum()))
display(gt["match_count"].describe(percentiles=[.5,.75,.9,.95,.99]))
display(gt["match_count"].value_counts().sort_index().rename_axis("matches").reset_index(name="s1_entities"))
display(pd.DataFrame({
    "category":["singleton","one_match","multi_match","has_S2","has_S3","has_both_S2_S3"],
    "count":[
        int(gt["match_count"].eq(0).sum()),
        int(gt["match_count"].eq(1).sum()),
        int(gt["match_count"].gt(1).sum()),
        int(gt["s2_count"].gt(0).sum()),
        int(gt["s3_count"].gt(0).sum()),
        int((gt["s2_count"].gt(0)&gt["s3_count"].gt(0)).sum())
    ]
}))

Ground-truth rows: 2206821
Ground-truth duplicate S1 IDs: 0


count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
50%      3.000000e+00
75%      5.000000e+00
90%      6.000000e+00
95%      6.000000e+00
99%      8.000000e+00
max      1.100000e+01
Name: match_count, dtype: float64

,matches,s1_entities
0,0,123247
1,1,119157
2,2,375212
3,3,530841
4,4,484115
5,5,321957
6,6,164868
7,7,63968
8,8,18680
9,9,4205


,category,count
0,singleton,123247
1,one_match,119157
2,multi_match,1964417
3,has_S2,1919076
4,has_S3,1940545
5,has_both_S2_S3,1776047


## 7. Sample labeled positive pairs without loading all source tables

In [12]:
positive_ids = []
seen = set()

for row in gt[["source1_entity_id","match_list"]].itertuples(index=False):
    for mid in row.match_list:
        key=(row.source1_entity_id,mid)
        if key not in seen:
            seen.add(key)
            positive_ids.append(key)

if len(positive_ids) > POSITIVE_PAIR_SAMPLE:
    rng=np.random.default_rng(SEED)
    take=rng.choice(len(positive_ids),POSITIVE_PAIR_SAMPLE,replace=False)
    positive_ids=[positive_ids[i] for i in take]

s1_needed={x[0] for x in positive_ids}
s2_needed={x[1] for x in positive_ids if x[1].startswith("S2-")}
s3_needed={x[1] for x in positive_ids if x[1].startswith("S3-")}

def retrieve_ids(path, needed):
    if not needed:
        return pd.DataFrame(columns=SOURCE_COLS)
    found=[]
    remaining=set(needed)
    for chunk in pd.read_csv(path,sep="\t",dtype=str,keep_default_na=False,na_filter=False,chunksize=CHUNK_SIZE):
        m=chunk["entity_id"].isin(remaining)
        if m.any():
            found.append(chunk.loc[m,SOURCE_COLS].copy())
            remaining -= set(chunk.loc[m,"entity_id"])
        if not remaining:
            break
    return pd.concat(found,ignore_index=True) if found else pd.DataFrame(columns=SOURCE_COLS)

t0=time.perf_counter()
s1_pos=retrieve_ids(paths["train_source1.tsv"],s1_needed)
s2_pos=retrieve_ids(paths["train_source2.tsv"],s2_needed)
s3_pos=retrieve_ids(paths["train_source3.tsv"],s3_needed)
print("Positive-pair retrieval minutes:",round((time.perf_counter()-t0)/60,2))
print("Retrieved:",len(s1_pos),len(s2_pos),len(s3_pos))

Positive-pair retrieval minutes: 0.66
Retrieved: 19931 9602 10398


In [13]:
def pair_key_features(a,b):
    an=translit_norm(a["business_name"])
    bn=translit_norm(b["business_name"])
    aa=translit_norm(a["business_address"])
    ba=translit_norm(b["business_address"])

    def sim(x,y,kind="ratio"):
        if HAS_RAPIDFUZZ:
            if kind=="w": return WRatio(x,y)
            if kind=="sort": return token_sort_ratio(x,y)
            if kind=="set": return token_set_ratio(x,y)
            return ratio(x,y)
        return 100*SequenceMatcher(None,x,y).ratio()

    return {
        "name_ratio":sim(an,bn),
        "name_wratio":sim(an,bn,"w"),
        "name_token_sort":sim(an,bn,"sort"),
        "name_token_set":sim(an,bn,"set"),
        "address_ratio":sim(aa,ba),
        "address_wratio":sim(aa,ba,"w"),
        "address_token_sort":sim(aa,ba,"sort"),
        "address_token_set":sim(aa,ba,"set"),
        "name_exact_translit":int(bool(an) and an==bn),
        "address_exact_translit":int(bool(aa) and aa==ba),
        "name_basic_exact":int(bool(basic_norm(a["business_name"])) and basic_norm(a["business_name"])==basic_norm(b["business_name"])),
        "address_basic_exact":int(bool(basic_norm(a["business_address"])) and basic_norm(a["business_address"])==basic_norm(b["business_address"])),
        "country_exact":int(bool(basic_norm(a["country"])) and basic_norm(a["country"])==basic_norm(b["country"])),
        "name_length_ratio":min(len(an),len(bn))/max(1,max(len(an),len(bn))),
        "address_length_ratio":min(len(aa),len(ba))/max(1,max(len(aa),len(ba))),
        "name_numeric_overlap":len(set(re.findall(r"\d+",an))&set(re.findall(r"\d+",bn)))/max(1,len(set(re.findall(r"\d+",an))|set(re.findall(r"\d+",bn)))),
        "address_numeric_overlap":len(set(re.findall(r"\d+",aa))&set(re.findall(r"\d+",ba)))/max(1,len(set(re.findall(r"\d+",aa))|set(re.findall(r"\d+",ba))))
    }

s1_lookup=s1_pos.set_index("entity_id").to_dict("index")
s2_lookup=s2_pos.set_index("entity_id").to_dict("index")
s3_lookup=s3_pos.set_index("entity_id").to_dict("index")

pos_features=[]
for sid,mid in positive_ids:
    if sid not in s1_lookup:
        continue
    target=s2_lookup.get(mid) if mid.startswith("S2-") else s3_lookup.get(mid)
    if target is None:
        continue
    f=pair_key_features(s1_lookup[sid],target)
    f.update({"s1_id":sid,"match_id":mid,"match_source":mid[:2]})
    pos_features.append(f)

positive_features=pd.DataFrame(pos_features)
print("Positive feature rows:",len(positive_features))
display(positive_features.groupby("match_source").mean(numeric_only=True).T if len(positive_features) else pd.DataFrame())

Positive feature rows: 20000


match_source,S2,S3
name_ratio,77.321294,79.255078
name_wratio,77.321294,79.255078
name_token_sort,77.321294,79.255078
name_token_set,77.321294,79.255078
address_ratio,78.683762,72.378152
address_wratio,78.683762,72.378152
address_token_sort,78.683762,72.378152
address_token_set,78.683762,72.378152
name_exact_translit,0.217142,0.216195
address_exact_translit,0.126536,0.043855


## 8. Hard-negative analysis from sample blocks

In [14]:
def build_sample_index(df, cols):
    out={}
    for c in cols:
        temp=df[df[c].astype(str).str.strip().ne("")]
        out[c]=temp.groupby(c)["entity_id"].apply(list).to_dict()
    return out

sample_indexes={}
for name in ["train_source2","train_source3"]:
    sample_indexes[name]=build_sample_index(
        norm_samples[name],
        ["business_name_translit","business_address_translit","business_name_basic","business_address_basic"]
    )

s1_sample=norm_samples["train_source1"].set_index("entity_id")
s2_sample=norm_samples["train_source2"].set_index("entity_id")
s3_sample=norm_samples["train_source3"].set_index("entity_id")

positive_set=set(positive_ids)
hard_neg=[]
for sid,row in s1_sample.iterrows():
    for target_name,target_df in [("train_source2",s2_sample),("train_source3",s3_sample)]:
        idx=sample_indexes[target_name]
        cand=set()
        for c in idx:
            value=str(row[c]).strip()
            if value:
                cand.update(idx[c].get(value,[]))
        for mid in cand:
            if (sid,mid) not in positive_set:
                hard_neg.append((sid,mid,target_name))
                if len(hard_neg)>=HARD_NEGATIVE_SAMPLE:
                    break
        if len(hard_neg)>=HARD_NEGATIVE_SAMPLE:
            break
    if len(hard_neg)>=HARD_NEGATIVE_SAMPLE:
        break

hard_rows=[]
for sid,mid,target_name in hard_neg:
    a=s1_sample.loc[sid]
    b=(s2_sample if target_name=="train_source2" else s3_sample).loc[mid]
    f=pair_key_features(a,b)
    f.update({"s1_id":sid,"match_id":mid,"target":target_name,"label":0})
    hard_rows.append(f)

hard_features=pd.DataFrame(hard_rows)
print("Hard negative feature rows:",len(hard_features))
display(hard_features.head(20))

Hard negative feature rows: 800


,name_ratio,name_wratio,name_token_sort,name_token_set,address_ratio,address_wratio,address_token_sort,address_token_set,name_exact_translit,address_exact_translit,name_basic_exact,address_basic_exact,country_exact,name_length_ratio,address_length_ratio,name_numeric_overlap,address_numeric_overlap,s1_id,match_id,target,label
0,100.000000,100.000000,100.000000,100.000000,31.578947,31.578947,31.578947,31.578947,1,0,0,0,1,1.000000,0.676471,0.0,0.0,S1-274123243,S2-441149697,train_source2,0
1,100.000000,100.000000,100.000000,100.000000,40.579710,40.579710,40.579710,40.579710,1,0,0,0,1,1.000000,0.971429,0.0,0.0,S1-274123243,S3-844434747,train_source3,0
2,100.000000,100.000000,100.000000,100.000000,38.636364,38.636364,38.636364,38.636364,1,0,1,0,1,1.000000,0.872340,0.0,0.0,S1-243240574,S3-456086219,train_source3,0
3,100.000000,100.000000,100.000000,100.000000,30.769231,30.769231,30.769231,30.769231,1,0,1,0,1,1.000000,0.911765,0.0,0.0,S1-654117873,S2-83920902,train_source2,0
4,100.000000,100.000000,100.000000,100.000000,36.666667,36.666667,36.666667,36.666667,1,0,1,0,1,1.000000,0.818182,0.0,0.0,S1-373189851,S2-865857304,train_source2,0
5,100.000000,100.000000,100.000000,100.000000,33.333333,33.333333,33.333333,33.333333,1,0,1,0,1,1.000000,0.833333,0.0,0.0,S1-956532047,S3-131108819,train_source3,0
6,100.000000,100.000000,100.000000,100.000000,17.721519,17.721519,17.721519,17.721519,1,0,0,0,1,1.000000,0.490566,0.0,0.0,S1-895947450,S3-847144383,train_source3,0
7,100.000000,100.000000,100.000000,100.000000,19.354839,19.354839,19.354839,19.354839,1,0,1,0,1,1.000000,0.823529,0.0,0.0,S1-762717156,S2-709539780,train_source2,0
8,100.000000,100.000000,100.000000,100.000000,20.588235,20.588235,20.588235,20.588235,1,0,1,0,1,1.000000,1.000000,0.0,0.0,S1-762717156,S3-45630009,train_source3,0
9,86.666667,86.666667,86.666667,86.666667,100.000000,100.000000,100.000000,100.000000,0,1,0,1,1,0.764706,1.000000,0.0,1.0,S1-77959673,S2-674745221,train_source2,0


## 9. Positive vs hard-negative signal comparison

In [15]:
if len(positive_features) and len(hard_features):
    pos=positive_features.copy()
    pos["label"]=1
    neg=hard_features.copy()
    compare=pd.concat([pos,neg],ignore_index=True)

    feature_cols=[
        "name_ratio","name_wratio","name_token_sort","name_token_set",
        "address_ratio","address_wratio","address_token_sort","address_token_set",
        "name_exact_translit","address_exact_translit",
        "name_basic_exact","address_basic_exact","country_exact",
        "name_length_ratio","address_length_ratio",
        "name_numeric_overlap","address_numeric_overlap"
    ]

    rows=[]
    for c in feature_cols:
        x=pd.to_numeric(compare[c],errors="coerce")
        p=x[compare["label"].eq(1)].mean()
        n=x[compare["label"].eq(0)].mean()
        rows.append({
            "feature":c,
            "positive_mean":p,
            "hard_negative_mean":n,
            "positive_median":x[compare["label"].eq(1)].median(),
            "hard_negative_median":x[compare["label"].eq(0)].median(),
            "mean_gap":p-n
        })
    display(pd.DataFrame(rows).sort_values("mean_gap",ascending=False))

    for c in ["name_ratio","address_ratio","name_token_set","address_token_set","country_exact"]:
        display(compare.groupby("label")[c].describe(percentiles=[.1,.25,.5,.75,.9]).assign(feature=c))

,feature,positive_mean,hard_negative_mean,positive_median,hard_negative_median,mean_gap
4,address_ratio,75.405475,33.317138,83.333333,28.232999,42.088337
6,address_token_sort,75.405475,33.317138,83.333333,28.232999,42.088337
7,address_token_set,75.405475,33.317138,83.333333,28.232999,42.088337
5,address_wratio,75.405475,33.317138,83.333333,28.232999,42.088337
16,address_numeric_overlap,0.706181,0.087741,1.000000,0.000000,0.618440
9,address_exact_translit,0.083550,0.040000,0.000000,0.000000,0.043550
11,address_basic_exact,0.075050,0.038750,0.000000,0.000000,0.036300
14,address_length_ratio,0.804691,0.776784,0.878788,0.820513,0.027906
15,name_numeric_overlap,0.014858,0.003750,0.000000,0.000000,0.011108
12,country_exact,1.000000,0.997500,1.000000,1.000000,0.002500


,count,mean,std,min,10%,25%,50%,75%,90%,max,feature
label,,,,,,,,,,,
0,800.0,98.319483,9.126705,0.0,97.289790,100.000000,100.000000,100.000000,100.0,100.0,name_ratio
1,20000.0,78.326668,26.871964,0.0,45.598086,72.413793,87.179487,96.296296,100.0,100.0,name_ratio


,count,mean,std,min,10%,25%,50%,75%,90%,max,feature
label,,,,,,,,,,,
0,800.0,33.317138,20.647373,0.0,17.541299,22.526408,28.232999,35.483871,51.343463,100.0,address_ratio
1,20000.0,75.405475,23.768986,0.0,44.822440,65.306122,83.333333,91.891892,97.709924,100.0,address_ratio


,count,mean,std,min,10%,25%,50%,75%,90%,max,feature
label,,,,,,,,,,,
0,800.0,98.319483,9.126705,0.0,97.289790,100.000000,100.000000,100.000000,100.0,100.0,name_token_set
1,20000.0,78.326668,26.871964,0.0,45.598086,72.413793,87.179487,96.296296,100.0,100.0,name_token_set


,count,mean,std,min,10%,25%,50%,75%,90%,max,feature
label,,,,,,,,,,,
0,800.0,33.317138,20.647373,0.0,17.541299,22.526408,28.232999,35.483871,51.343463,100.0,address_token_set
1,20000.0,75.405475,23.768986,0.0,44.822440,65.306122,83.333333,91.891892,97.709924,100.0,address_token_set


,count,mean,std,min,10%,25%,50%,75%,90%,max,feature
label,,,,,,,,,,,
0,800.0,0.9975,0.049969,0.0,1.0,1.0,1.0,1.0,1.0,1.0,country_exact
1,20000.0,1.0000,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,country_exact


## 10. Train/test drift from full-data counts + samples

In [16]:
drift_rows=[]

for source_id in [1,2,3]:
    tr=profiles[f"train_source{source_id}"]
    te=profiles[f"test_source{source_id}"]

    for metric in ["blank_name_pct","blank_address_pct","blank_country_pct","name_mean_len","address_mean_len"]:
        tr_value = {
            "blank_name_pct":100*tr["blank"]["business_name"]/max(1,tr["rows"]),
            "blank_address_pct":100*tr["blank"]["business_address"]/max(1,tr["rows"]),
            "blank_country_pct":100*tr["blank"]["country"]/max(1,tr["rows"]),
            "name_mean_len":tr["name_len_sum"]/max(1,tr["rows"]),
            "address_mean_len":tr["address_len_sum"]/max(1,tr["rows"])
        }[metric]
        te_value = {
            "blank_name_pct":100*te["blank"]["business_name"]/max(1,te["rows"]),
            "blank_address_pct":100*te["blank"]["business_address"]/max(1,te["rows"]),
            "blank_country_pct":100*te["blank"]["country"]/max(1,te["rows"]),
            "name_mean_len":te["name_len_sum"]/max(1,te["rows"]),
            "address_mean_len":te["address_len_sum"]/max(1,te["rows"])
        }[metric]
        drift_rows.append({
            "source":source_id,
            "metric":metric,
            "train":tr_value,
            "test":te_value,
            "delta":te_value-tr_value
        })

display(pd.DataFrame(drift_rows))

for source_id in [1,2,3]:
    tr=set(profiles[f"train_source{source_id}"]["country_counts"])
    te=set(profiles[f"test_source{source_id}"]["country_counts"])
    print("Source",source_id,"test-only country labels:",sorted(te-tr)[:100])

,source,metric,train,test,delta
0,1,blank_name_pct,0.000000,0.000000,0.000000
1,1,blank_address_pct,0.000000,0.000000,0.000000
2,1,blank_country_pct,0.000000,0.000000,0.000000
3,1,name_mean_len,24.034403,23.836426,-0.197977
4,1,address_mean_len,52.066213,57.213180,5.146967
5,2,blank_name_pct,0.000000,0.000000,0.000000
6,2,blank_address_pct,3.356105,2.647857,-0.708248
7,2,blank_country_pct,0.000000,0.000000,0.000000
8,2,name_mean_len,25.103575,25.703646,0.600071
9,2,address_mean_len,46.225909,50.413406,4.187497


Source 1 test-only country labels: ['france']
Source 2 test-only country labels: ['france']
Source 3 test-only country labels: ['france']


## 11. Sampled source overlap and ambiguity diagnostics

In [17]:
overlap_rows=[]

for key in ["business_name_translit","business_address_translit"]:
    maps={}
    for src in ["train_source1","train_source2","train_source3"]:
        maps[src]=set(norm_samples[src][key].astype(str).loc[lambda s:s.str.strip().ne("")])
    for a,b in [("train_source1","train_source2"),("train_source1","train_source3")]:
        inter=maps[a]&maps[b]
        overlap_rows.append({
            "representation":key,
            "source_a":a,
            "source_b":b,
            "a_unique_sample":len(maps[a]),
            "b_unique_sample":len(maps[b]),
            "shared_values":len(inter),
            "shared_pct_of_a":100*len(inter)/max(1,len(maps[a])),
            "shared_pct_of_b":100*len(inter)/max(1,len(maps[b]))
        })

display(pd.DataFrame(overlap_rows))

ambiguity_rows=[]
for src in ["train_source1","train_source2","train_source3","test_source1","test_source2","test_source3"]:
    df=norm_samples[src]
    for field in ["business_name_translit","business_address_translit"]:
        counts=df[field].value_counts()
        counts=counts[(counts.index.astype(str).str.strip() != "")]
        ambiguity_rows.append({
            "dataset":src,
            "field":field,
            "sample_rows":len(df),
            "values_unique":len(counts),
            "rows_in_collision_2plus":int(counts[counts>=2].sum()),
            "rows_in_collision_5plus":int(counts[counts>=5].sum()),
            "max_collision":int(counts.max()) if len(counts) else 0
        })

display(pd.DataFrame(ambiguity_rows))

,representation,source_a,source_b,a_unique_sample,b_unique_sample,shared_values,shared_pct_of_a,shared_pct_of_b
0,business_name_translit,train_source1,train_source2,19425,18030,256,1.317889,1.419856
1,business_name_translit,train_source1,train_source3,19425,18899,278,1.431145,1.470977
2,business_address_translit,train_source1,train_source2,19990,19316,22,0.110055,0.113895
3,business_address_translit,train_source1,train_source3,19990,19337,10,0.050025,0.051714


,dataset,field,sample_rows,values_unique,rows_in_collision_2plus,rows_in_collision_5plus,max_collision
0,train_source1,business_name_translit,20000,19425,1077,5,5
1,train_source1,business_address_translit,20000,19990,20,0,2
2,train_source2,business_name_translit,20000,18030,169,6,6
3,train_source2,business_address_translit,20000,19316,10,0,2
4,train_source3,business_name_translit,20000,18899,292,16,11
5,train_source3,business_address_translit,20000,19337,18,0,2
6,test_source1,business_name_translit,17326,16906,803,0,4
7,test_source1,business_address_translit,17326,17304,43,0,3
8,test_source2,business_name_translit,20000,17763,106,0,4
9,test_source2,business_address_translit,20000,19412,28,0,2


## 12. Blocking-style diagnostics on the sample

In [18]:
def gt_links_by_target(source_prefix):
    result={}
    for row in gt.itertuples(index=False):
        mids=[x for x in row.match_list if x.startswith(source_prefix)]
        if mids:
            result[row.source1_entity_id]=set(mids)
    return result

def block_recall_sample(target_name, target_prefix):
    if target_name not in norm_samples or len(norm_samples[target_name])==0:
        return pd.DataFrame()

    target=norm_samples[target_name]
    idx={}
    for col in ["business_name_translit","business_address_translit","business_name_basic","business_address_basic"]:
        idx[col]=target.groupby(col)["entity_id"].apply(list).to_dict()

    truth=gt_links_by_target(target_prefix)
    s1=norm_samples["train_source1"]
    eligible=s1[s1["entity_id"].isin(truth)]

    configs=[
        ["business_name_translit"],
        ["business_address_translit"],
        ["business_name_translit","business_address_translit"],
        ["business_name_basic","business_address_basic"],
        ["business_name_translit","business_address_translit","business_name_basic","business_address_basic"]
    ]

    out=[]
    for cols in configs:
        pair_hits=0
        pair_total=0
        entity_hits=0
        entity_total=0
        candidate_sizes=[]

        for r in eligible.itertuples(index=False):
            true=set(truth[r.entity_id])
            cand=set()
            for c in cols:
                v=str(getattr(r,c)).strip()
                if v:
                    cand.update(idx[c].get(v,[]))
            pair_total+=len(true)
            pair_hits+=len(true&cand)
            entity_total+=1
            entity_hits+=int(true.issubset(cand))
            candidate_sizes.append(len(cand))

        out.append({
            "target":target_prefix,
            "blocking":" + ".join(cols),
            "eligible_s1_in_sample":entity_total,
            "pair_recall":pair_hits/max(1,pair_total),
            "all_links_covered_pct":100*entity_hits/max(1,entity_total),
            "median_candidates":np.median(candidate_sizes) if candidate_sizes else 0,
            "p95_candidates":np.quantile(candidate_sizes,.95) if candidate_sizes else 0,
            "max_candidates":np.max(candidate_sizes) if candidate_sizes else 0
        })

    return pd.DataFrame(out)

display(block_recall_sample("train_source2","S2"))
display(block_recall_sample("train_source3","S3"))

,target,blocking,eligible_s1_in_sample,pair_recall,all_links_covered_pct,median_candidates,p95_candidates,max_candidates
0,S2,business_name_translit,17403,0.000658,0.028731,0.0,0.0,3
1,S2,business_address_translit,17403,0.000598,0.034477,0.0,0.0,1
2,S2,business_name_translit + business_address_translit,17403,0.001106,0.040223,0.0,0.0,3
3,S2,business_name_basic + business_address_basic,17403,0.001046,0.040223,0.0,0.0,2
4,S2,business_name_translit + business_address_translit + business_name_basic + business_address_basic,17403,0.001196,0.040223,0.0,0.0,3


,target,blocking,eligible_s1_in_sample,pair_recall,all_links_covered_pct,median_candidates,p95_candidates,max_candidates
0,S3,business_name_translit,17605,0.000725,0.028401,0.0,0.0,3
1,S3,business_address_translit,17605,0.000195,0.005680,0.0,0.0,1
2,S3,business_name_translit + business_address_translit,17605,0.000920,0.034081,0.0,0.0,3
3,S3,business_name_basic + business_address_basic,17605,0.000781,0.028401,0.0,0.0,3
4,S3,business_name_translit + business_address_translit + business_name_basic + business_address_basic,17605,0.000948,0.034081,0.0,0.0,3


## 13. Train/test script and length comparison

In [19]:
comparison_rows=[]

for source_id in [1,2,3]:
    for field in ["business_name","business_address"]:
        tr=norm_samples[f"train_source{source_id}"][field].astype(str).str.len()
        te=norm_samples[f"test_source{source_id}"][field].astype(str).str.len()
        comparison_rows.append({
            "source":source_id,
            "field":field,
            "train_sample_n":len(tr),
            "test_sample_n":len(te),
            "train_mean_len":tr.mean(),
            "test_mean_len":te.mean(),
            "train_p50":tr.quantile(.5),
            "test_p50":te.quantile(.5),
            "train_p95":tr.quantile(.95),
            "test_p95":te.quantile(.95),
            "train_max":tr.max(),
            "test_max":te.max()
        })

display(pd.DataFrame(comparison_rows))

script_drift=[]
for source_id in [1,2,3]:
    for field in ["business_name","business_address"]:
        tr=threaded_metrics(sample_dfs[f"train_source{source_id}"][field].tolist())
        te=threaded_metrics(sample_dfs[f"test_source{source_id}"][field].tolist())
        for script in sorted(set(tr["dominant_script"])|set(te["dominant_script"])):
            script_drift.append({
                "source":source_id,
                "field":field,
                "script":script,
                "train_pct":100*(tr["dominant_script"].eq(script)).mean(),
                "test_pct":100*(te["dominant_script"].eq(script)).mean()
            })
display(pd.DataFrame(script_drift))

,source,field,train_sample_n,test_sample_n,train_mean_len,test_mean_len,train_p50,test_p50,train_p95,test_p95,train_max,test_max
0,1,business_name,20000,17326,24.11510,23.818019,24.0,23.0,37.0,36.0,87,92
1,1,business_address,20000,17326,51.93395,56.982570,41.0,50.0,103.0,104.0,194,186
2,2,business_name,20000,20000,25.14035,25.707500,25.0,25.0,40.0,42.0,69,74
3,2,business_address,20000,20000,46.30200,50.277800,37.0,42.0,96.0,98.0,199,226
4,3,business_name,20000,20000,25.27820,25.686100,25.0,25.0,42.0,42.0,78,92
5,3,business_address,20000,20000,46.72640,48.580550,42.0,43.0,91.0,94.0,178,189


,source,field,script,train_pct,test_pct
0,1,business_name,LATIN,100.000,100.000
1,1,business_address,LATIN,100.000,100.000
2,2,business_name,BENGALI,0.610,0.820
3,2,business_name,DEVANAGARI,5.610,6.090
4,2,business_name,GUJARATI,0.500,0.765
5,2,business_name,GURMUKHI,0.145,0.130
6,2,business_name,KANNADA,0.670,0.930
7,2,business_name,LATIN,90.395,88.925
8,2,business_name,MALAYALAM,0.365,0.400
9,2,business_name,NONE,0.005,0.000


## 14. Final EDA focus report

In [20]:
focus=[]

for name,p in profiles.items():
    n=p["rows"]
    if 100*p["blank"]["business_name"]/max(1,n) > 1:
        focus.append({"priority":"high","dataset":name,"area":"missing_name","observation":f'{100*p["blank"]["business_name"]/max(1,n):.2f}% blank names'})
    if 100*p["blank"]["business_address"]/max(1,n) > 1:
        focus.append({"priority":"high","dataset":name,"area":"missing_address","observation":f'{100*p["blank"]["business_address"]/max(1,n):.2f}% blank addresses'})
    if p["wrong_prefix"] > 0:
        focus.append({"priority":"high","dataset":name,"area":"ID_integrity","observation":f'{p["wrong_prefix"]} IDs have unexpected prefixes'})
    if p["duplicate_rows_within_chunks"] > 0:
        focus.append({"priority":"medium","dataset":name,"area":"duplicate_rows","observation":f'{p["duplicate_rows_within_chunks"]} duplicate rows detected within chunks'})
    if p["id_duplicate_within_chunks"] > 0:
        focus.append({"priority":"high","dataset":name,"area":"duplicate_ID","observation":f'{p["id_duplicate_within_chunks"]} duplicate IDs detected within chunks'})

edge_counts=pd.DataFrame(edge_summary)
if len(edge_counts):
    for _,r in edge_counts[edge_counts["sample_pct"]>=1].sort_values("sample_pct",ascending=False).head(30).iterrows():
        focus.append({"priority":"medium","dataset":r["dataset"],"area":r["issue"],"observation":f'{r["sample_pct"]:.2f}% of the deep sample'})

if len(gt):
    singleton=100*gt["match_count"].eq(0).mean()
    multi=100*gt["match_count"].gt(1).mean()
    if singleton>0:
        focus.append({"priority":"high","dataset":"ground_truth","area":"singletons","observation":f"{singleton:.2f}% of S1 entities have no matches"})
    if multi>0:
        focus.append({"priority":"high","dataset":"ground_truth","area":"multi_match","observation":f"{multi:.2f}% of S1 entities have multiple matches"})

test_only=[]
for source_id in [1,2,3]:
    tr=set(profiles[f"train_source{source_id}"]["country_counts"])
    te=set(profiles[f"test_source{source_id}"]["country_counts"])
    extra=sorted(te-tr)
    if extra:
        test_only.append({"priority":"high","dataset":f"test_source{source_id}","area":"unseen_country_labels","observation":str(extra[:50])})
focus.extend(test_only)

focus_df=pd.DataFrame(focus)
if len(focus_df):
    order={"high":0,"medium":1,"low":2}
    focus_df["order"]=focus_df["priority"].map(order)
    focus_df=focus_df.sort_values(["order","dataset","area"]).drop(columns="order").reset_index(drop=True)
display(focus_df)

outdir=Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
focus_df.to_csv(outdir/"eda_focus_report.csv",index=False)
full_profile.to_csv(outdir/"eda_full_profile.csv",index=False)
if len(positive_features):
    positive_features.to_csv(outdir/"eda_positive_pair_features.csv",index=False)
if len(hard_features):
    hard_features.to_csv(outdir/"eda_hard_negative_features.csv",index=False)

print("Saved EDA reports to:", outdir)

,priority,dataset,area,observation
0,high,ground_truth,multi_match,89.02% of S1 entities have multiple matches
1,high,ground_truth,singletons,5.58% of S1 entities have no matches
2,high,test_source1,unseen_country_labels,['france']
3,high,test_source2,missing_address,2.65% blank addresses
4,high,test_source2,unseen_country_labels,['france']
5,high,test_source3,missing_address,2.68% blank addresses
6,high,test_source3,unseen_country_labels,['france']
7,high,train_source2,missing_address,3.36% blank addresses
8,high,train_source3,missing_address,3.33% blank addresses
9,medium,test_source2,address_missing,2.87% of the deep sample


Saved EDA reports to: /kaggle/working


## Interpretation

Use the outputs to identify data problems and feature-engineering focus areas only.

Pay particular attention to:
- missing-name/address rates
- cross-script and mixed-script prevalence
- transliteration collisions
- generic/high-frequency names or addresses
- unusual Unicode/control characters
- very short and very long fields
- train/test drift
- unseen test country labels
- singleton and multi-match prevalence
- positive-pair cases where raw/baseline lexical similarity is weak
- hard negatives with deceptively strong lexical similarity
- blocking candidates that become very large or miss known positive links

The notebook does not select a model or prescribe a final pipeline.